# TMF Classifier — BioClinicalBERT training in Colab

This notebook trains the current three-class TMF classifier from the locally generated `train.csv` and `test.csv` files. In Colab, select **Runtime → Change runtime type → T4 GPU** before running.

Expected labels: `protocol`, `safety_report`, and `statistical_analysis_plan`.

In [ ]:
# Colab includes PyTorch. Install the remaining training dependencies.
!pip -q install -U transformers datasets accelerate scikit-learn joblib tqdm

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from google.colab import files

# Set this to True only when your project lives in Google Drive.
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/TMF_Classifier')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ARTIFACT_DIR = DRIVE_PROJECT_DIR / 'artifacts'
    TRAIN_PATH = ARTIFACT_DIR / 'train.csv'
    TEST_PATH = ARTIFACT_DIR / 'test.csv'
else:
    # Upload artifacts/train.csv and artifacts/test.csv when prompted.
    ARTIFACT_DIR = Path('/content/tmf_artifacts')
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    TRAIN_PATH = ARTIFACT_DIR / 'train.csv'
    TEST_PATH = ARTIFACT_DIR / 'test.csv'
    if not (TRAIN_PATH.exists() and TEST_PATH.exists()):
        print('Upload train.csv and test.csv from your local artifacts/ folder.')
        uploaded = files.upload()
        for filename, content in uploaded.items():
            destination = ARTIFACT_DIR / Path(filename).name
            destination.write_bytes(content)

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Both train.csv and test.csv must be present in the artifact directory.')

print(f'Train data: {TRAIN_PATH}')
print(f'Test data:  {TEST_PATH}')
print(f'Outputs:    {ARTIFACT_DIR}')

In [ ]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
# Change NUM_EPOCHS here when you want more/less training epochs.
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
MAX_LENGTH = 512
RANDOM_SEED = 42

MODEL_DIR = ARTIFACT_DIR / 'saved_bioclinicalbert_tmf_3class'
TRAINING_DIR = ARTIFACT_DIR / 'training_results'
LABEL_ENCODER_PATH = ARTIFACT_DIR / 'label_encoder.pkl'

train_df = pd.read_csv(TRAIN_PATH).dropna(subset=['chunk_text', 'class']).copy()
test_df = pd.read_csv(TEST_PATH).dropna(subset=['chunk_text', 'class']).copy()
train_df['chunk_text'] = train_df['chunk_text'].astype(str)
test_df['chunk_text'] = test_df['chunk_text'].astype(str)

label_encoder = LabelEncoder()
label_encoder.fit(train_df['class'].astype(str))
unknown_labels = set(test_df['class'].astype(str)).difference(label_encoder.classes_)
if unknown_labels:
    raise ValueError(f'Test labels absent from training data: {sorted(unknown_labels)}')

train_df['labels'] = label_encoder.transform(train_df['class'].astype(str))
test_df['labels'] = label_encoder.transform(test_df['class'].astype(str))
joblib.dump(label_encoder, LABEL_ENCODER_PATH)

print('Classes:', label_encoder.classes_.tolist())
print('Train chunks:', len(train_df), '| Test chunks:', len(test_df))
print('Train documents:', train_df['file_name'].nunique(), '| Test documents:', test_df['file_name'].nunique())

In [ ]:
import inspect
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(RANDOM_SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    id2label={index: label for index, label in enumerate(label_encoder.classes_)},
    label2id={label: index for index, label in enumerate(label_encoder.classes_)},
)

def tokenize(batch):
    return tokenizer(batch['chunk_text'], truncation=True, max_length=MAX_LENGTH)

train_dataset = Dataset.from_pandas(train_df[['chunk_text', 'labels']], preserve_index=False).map(tokenize, batched=True)
test_dataset = Dataset.from_pandas(test_df[['chunk_text', 'labels']], preserve_index=False).map(tokenize, batched=True)

# Required to avoid torchvision formatter errors.
torch_columns = ['input_ids', 'attention_mask', 'labels']
train_dataset.set_format(type='torch', columns=torch_columns)
test_dataset.set_format(type='torch', columns=torch_columns)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(labels, predictions, average='macro', zero_division=0)),
    }

training_kwargs = {
    'output_dir': str(TRAINING_DIR),
    'learning_rate': LEARNING_RATE,
    'per_device_train_batch_size': TRAIN_BATCH_SIZE,
    'per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'num_train_epochs': NUM_EPOCHS,
    'save_strategy': 'epoch',
    'logging_strategy': 'steps',
    'logging_steps': 10,
    'report_to': 'none',
    'disable_tqdm': False,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'macro_f1',
    'greater_is_better': True,
}
argument_parameters = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs['eval_strategy' if 'eval_strategy' in argument_parameters else 'evaluation_strategy'] = 'epoch'
training_args = TrainingArguments(**training_kwargs)

# Do not pass tokenizer=tokenizer here: newer Transformers releases may reject it.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f'Saved model and tokenizer to {MODEL_DIR}')

In [ ]:
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

predicted_indices = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)
test_results = test_df.copy()
test_results['predicted_class'] = label_encoder.inverse_transform(predicted_indices)
classes = label_encoder.classes_.tolist()

def summarize(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=classes, average='macro', zero_division=0)),
        'classification_report': classification_report(
            y_true, y_pred, labels=classes, target_names=classes, zero_division=0, output_dict=True
        ),
    }

chunk_metrics = summarize(test_results['class'], test_results['predicted_class'])
document_results = test_results.groupby('file_name', as_index=False).agg(
    actual_class=('class', lambda values: Counter(values).most_common(1)[0][0]),
    predicted_class=('predicted_class', lambda values: Counter(values).most_common(1)[0][0]),
)
document_metrics = summarize(document_results['actual_class'], document_results['predicted_class'])

chunk_matrix = confusion_matrix(test_results['class'], test_results['predicted_class'], labels=classes)
document_matrix = confusion_matrix(document_results['actual_class'], document_results['predicted_class'], labels=classes)
matrix_rows = []
for level, matrix in [('chunk_level', chunk_matrix), ('document_level', document_matrix)]:
    for true_label, row in zip(classes, matrix):
        matrix_rows.append({'evaluation_level': level, 'true_label': true_label, **dict(zip(classes, row.tolist()))})

metrics = {'chunk_level': chunk_metrics, 'document_level': document_metrics}
metadata = {
    'train_docs': int(train_df['file_name'].nunique()),
    'test_docs': int(test_df['file_name'].nunique()),
    'train_chunks': int(len(train_df)),
    'test_chunks': int(len(test_df)),
    'classes': classes,
    'model_name': MODEL_NAME,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'chunk_level_metrics': chunk_metrics,
    'document_level_metrics': document_metrics,
}

(ARTIFACT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(ARTIFACT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
pd.DataFrame(matrix_rows).to_csv(ARTIFACT_DIR / 'confusion_matrix.csv', index=False)
test_results.to_csv(ARTIFACT_DIR / 'test_predictions.csv', index=False)

print('Chunk-level:', chunk_metrics['accuracy'], 'accuracy |', chunk_metrics['macro_f1'], 'macro F1')
print('Document-level:', document_metrics['accuracy'], 'accuracy |', document_metrics['macro_f1'], 'macro F1')

In [ ]:
# Download the trained model, label encoder, metrics, and prediction files.
import shutil

archive_path = shutil.make_archive('/content/tmf_classifier_artifacts', 'zip', ARTIFACT_DIR)
print(f'Created {archive_path}')
files.download(archive_path)